# Actividad 

## Analítica de Datos

### Pregunta orientadora

> **¿Qué tan diferentes son los pingüinos registrados en la base Palmer Penguins y qué decisiones de limpieza, transformación e inferencia permiten describir esas diferencias con evidencia?**

Esta actividad integra los temas trabajados en las clases anteriores:

- auditoría y limpieza de datos;
- valores faltantes;
- detección y tratamiento responsable de valores atípicos;
- transformación de variables continuas;
- transformación de variables cualitativas;
- normalidad;
- intervalos de confianza;
- pruebas de hipótesis;
- interpretación de resultados.

> **Idea central:** no se busca aplicar técnicas mecánicamente. Cada decisión debe justificarse a partir de la pregunta, el tipo de variable y la estructura de la base.

Al finalizar, cada grupo debe entregar:

1. una base de trabajo limpia;
2. evidencia gráfica y numérica de la revisión de atípicos;
3. al menos una transformación continua y una categórica;
4. un intervalo de confianza correctamente interpretado;
5. una prueba de hipótesis correctamente formulada e interpretada;
6. una prueba de normalidad acompañada por un gráfico Q-Q;
7. una conclusión de máximo 8 líneas basada en evidencia.

# 1. Base de datos: Palmer Penguins

Usaremos la base **`penguins`**, disponible a través de `seaborn`.

Cada fila corresponde a un pingüino observado. La base contiene mediciones morfológicas y variables categóricas que permiten comparar grupos.

## Variables

| Variable | Tipo | Descripción |
|---|---|---|
| `species` | Cualitativa nominal | Especie: Adelie, Chinstrap o Gentoo |
| `island` | Cualitativa nominal | Isla donde fue observado el pingüino |
| `bill_length_mm` | Cuantitativa continua | Longitud del pico en milímetros |
| `bill_depth_mm` | Cuantitativa continua | Profundidad del pico en milímetros |
| `flipper_length_mm` | Cuantitativa continua | Longitud de la aleta en milímetros |
| `body_mass_g` | Cuantitativa continua | Masa corporal en gramos |
| `sex` | Cualitativa nominal | Sexo registrado del pingüino |

Esta base es especialmente útil porque combina:

- variables continuas y cualitativas;
- valores faltantes reales;
- grupos naturalmente diferenciados;
- variables adecuadas para análisis descriptivo e inferencial.

## Preguntas de análisis

Durante la actividad intentaremos responder, entre otras, las siguientes preguntas:

1. ¿La base presenta problemas de calidad que deban corregirse?
2. ¿Existen observaciones potencialmente atípicas?
3. ¿Es razonable estudiar la masa corporal sin distinguir por especie?
4. ¿Cuál es la masa corporal media de los pingüinos Gentoo y con qué incertidumbre se estima?
5. Entre pingüinos Adelie, ¿los machos presentan mayor masa corporal media que las hembras?
6. ¿La variable `body_mass_g` puede considerarse aproximadamente normal?

# 2. Preparación del entorno

In [1]:
%mamba install numpy pandas matplotlib seaborn scikit-learn

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, seaborn, scikit-learn
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 3.117 seconds
  Name                Version    Build                Channel
---------------------------------------------------------------------------
+ brotli-python       1.2.0      py313ha26e73d_2      emscripten-forge-4x
+ certifi             2026.7.22  pyhd8ed1ab_0         conda-forge
+ charset-normalizer  3.5.1      pyhd8ed1ab_0         conda-forge
+ idna                3.19       pyhcf101f3_0         conda-forge
+ joblib              1.5.3      py313h1804a44_3      emscripten-forge-4x
+ narwhals            2.24.0     pyhcf101f3_0         conda-forge
+ pandas              3.0.5      np23py313h1e705a5_0  emscripten-forge-4x
+ patsy               1.0.2      py313h1804a44_3      emscripten-forge-4x
+ pysocks             1.7.1      py313h1804a44_3      emscripten-forge-4x
+ p

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

np.random.seed(42)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

print("Entorno preparado.")

Entorno preparado.


## 2.1 Carga de la base

El notebook intenta primero leer un archivo local `penguins.csv`, si está disponible. En caso contrario, carga la base directamente desde Seaborn.

In [3]:
candidatos = [
    Path("penguins.csv"),
    Path.cwd() / "penguins.csv",
    Path("/mnt/data") / "penguins.csv"
]

ruta_local = next((ruta for ruta in candidatos if ruta.exists()), None)

if ruta_local is not None:
    penguins_raw = pd.read_csv(ruta_local)
    print(f"Base cargada desde: {ruta_local}")
else:
    penguins_raw = sns.load_dataset("penguins")
    print("Base cargada con sns.load_dataset('penguins').")

penguins = penguins_raw.copy()
penguins.head()

Base cargada desde: penguins.csv


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.100,18.700,181.000,"3,750.000",MALE
1,Adelie,Torgersen,39.500,17.400,186.000,"3,800.000",FEMALE
2,Adelie,Torgersen,40.300,18.000,195.000,"3,250.000",FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.700,19.300,193.000,"3,450.000",FEMALE


# PARTE I. Comprender, auditar y limpiar

## 3. Auditoría inicial

### Actividad 1 — Comprensión de la estructura

Realice una auditoría inicial de la base.

Debe identificar:

- número de filas y columnas;
- nombres de las variables;
- tipos de datos;
- resumen descriptivo;
- categorías presentes en `species`, `island` y `sex`;
- cantidad y porcentaje de valores faltantes;
- filas duplicadas.

### Pregunta

> ¿Qué problemas requieren realmente una acción de limpieza y cuáles son simplemente características naturales de la base?

In [ ]:
# TODO 1. Muestre dimensiones, tipos y resumen descriptivo.
# Sugerencias:
# penguins.shape
# penguins.info()
# penguins.describe(include="all").T

In [ ]:
# TODO 2. Revise las categorías de las variables cualitativas.
# Variables: species, island, sex

In [ ]:
# TODO 3. Construya una tabla con:
# - número de faltantes
# - porcentaje de faltantes
# para cada variable.

In [ ]:
# TODO 4. Determine si existen filas duplicadas.

## 4. Decisión de limpieza

No toda ausencia debe imputarse automáticamente.

Para esta actividad:

- conserve una copia de la base original;
- documente cualquier eliminación;
- para análisis inferenciales sobre `body_mass_g` y `sex`, trabaje únicamente con observaciones que tengan información disponible en las variables necesarias;
- **no impute `body_mass_g` antes de una prueba de hipótesis**, porque una imputación artificial puede modificar medias, variabilidad y p-valores;
- si necesita codificar una variable categórica para un flujo de modelado, una categoría faltante podría tratarse de otra manera, pero debe justificarse.

### Actividad 2 — Construya `penguins_clean`

Cree una base limpia para los análisis posteriores.

Explique en una celda Markdown:

1. qué filas eliminó, si eliminó alguna;
2. qué faltantes decidió conservar temporalmente;
3. por qué su estrategia es adecuada para los análisis que realizará.

In [ ]:
# TODO 5. Cree la base de trabajo.
penguins_clean = penguins.copy()

# Complete aquí su estrategia de limpieza.

# PARTE II. Valores atípicos

## 5. Detección de posibles outliers

Un valor atípico no es automáticamente un error.

Utilice el criterio IQR:

$$IQR = Q_3-Q_1,$$

$$
L_I = Q_1-1.5IQR,
\qquad
L_S = Q_3+1.5IQR.
$$

y el **$z$-score**.

### Actividad 3

Para `body_mass_g`:

1. calcule $Q_1$, $Q_3$, $IQR$ y límites de Tukey;
2. cuente las observaciones detectadas como atípicas;
3. construya un boxplot general;
4. construya un boxplot de `body_mass_g` **por especie**.

Después responda:

> Si un valor parece extremo en la base completa pero es coherente dentro de su especie, ¿debería eliminarse?

In [ ]:
# TODO 6. Detección IQR para body_mass_g.

x = penguins_clean["body_mass_g"].dropna()

# q1 = ...
# q3 = ...
# iqr = ...
# limite_inferior = ...
# limite_superior = ...
# outliers = ...

# print(...)

In [ ]:
# TODO 7. Construya:
# 1) boxplot general de body_mass_g
# 2) boxplot de body_mass_g por species

### Extensión breve

Repita el conteo IQR para:

- `bill_length_mm`;
- `bill_depth_mm`;
- `flipper_length_mm`.

No elimine observaciones solo porque un algoritmo las marque. Primero determine si son plausibles en contexto.

In [ ]:
# TODO 8. Cree una función o un ciclo que reporte el número de outliers IQR
# para las cuatro variables continuas.

# PARTE III. Transformación de variables

## 6. Variables continuas

Una transformación de escala no implica normalidad.

### Actividad 4A — Estandarización

Estandarice:

- `bill_length_mm`;
- `body_mass_g`.

Utilice:

$$
z=\frac{x-\bar{x}}{s}.
$$

Compruebe que las variables transformadas tienen aproximadamente:

$$
\text{media}=0,
\qquad
\text{desviación}=1.
$$

Compare además la asimetría antes y después de estandarizar.

### Pregunta

> ¿La estandarización cambió la forma de la distribución?

In [ ]:
# TODO 9. Use StandardScaler sobre bill_length_mm y body_mass_g.
# Trabaje únicamente con filas completas para estas dos variables.

continuas = ["bill_length_mm", "body_mass_g"]

# Complete aquí.

## 7. Variables cualitativas

`species`, `island` y `sex` son variables **nominales**: sus categorías no tienen un orden numérico natural.

### Actividad 4B — One-Hot Encoding

Aplique `OneHotEncoder` a:

- `species`;
- `island`;
- `sex`.

Use:

```python
OneHotEncoder(handle_unknown="ignore", sparse_output=False)
```

Muestre:

1. las categorías aprendidas;
2. los nombres de las columnas generadas;
3. las primeras cinco filas codificadas.

### Pregunta

> ¿Por qué no sería apropiado codificar `species` como Adelie = 1, Chinstrap = 2, Gentoo = 3 y tratar esos números como magnitudes?

In [ ]:
# TODO 10. One-Hot Encoding.
categoricas = ["species", "island", "sex"]

# Complete aquí.

# PARTE IV. Intervalos de confianza

## 8. IC del 95% para una media

Nos concentraremos en los pingüinos **Gentoo**.

Sea:

$
\mu_G=\text{masa corporal media poblacional de pingüinos Gentoo}.
$

Construya un intervalo de confianza del 95%:

$
\bar{x}
\pm
t_{\alpha/2,n-1}
\frac{s}{\sqrt{n}}.
$

### Actividad 5

1. filtre `body_mass_g` para `species == "Gentoo"`;
2. calcule $n$, media, desviación y error estándar;
3. construya el IC del 95%;
4. interprete el resultado **en unidades de gramos**.

La interpretación debe referirse al parámetro poblacional y no únicamente a la muestra.

In [ ]:
# TODO 11. IC 95% para la media de body_mass_g de Gentoo.

gentoo_mass = penguins_clean.loc[
    penguins_clean["species"] == "Gentoo",
    "body_mass_g"
].dropna()

alpha = 0.05

# Complete aquí.

# PARTE V. Pruebas de hipótesis y normalidad

## 9. Comparación de dos medias independientes

Para evitar mezclar especies con tamaños corporales muy distintos, trabaje únicamente con **Adelie**.

### Pregunta inferencial

> Entre los pingüinos Adelie, ¿la masa corporal media de los machos es mayor que la de las hembras?

Defina:

$
\mu_M=\text{masa corporal media de machos Adelie},
$

$
\mu_F=\text{masa corporal media de hembras Adelie}.
$

Formule una prueba unilateral:

$
H_0:\mu_M-\mu_F\leq 0,
$

$
H_1:\mu_M-\mu_F>0.
$

Utilice una **t de Welch** con:

$
\alpha=0.05.
$

### Actividad 6

1. construya las dos muestras;
2. reporte tamaños, medias y desviaciones estándar;
3. ejecute la prueba t de Welch;
4. reporte estadístico y p-valor;
5. tome una decisión;
6. interprete en contexto.

> Recuerde: significancia estadística no equivale a causalidad.

In [ ]:
# TODO 12. Prueba t de Welch unilateral.

adelie_male = penguins_clean.loc[
    (penguins_clean["species"] == "Adelie") &
    (penguins_clean["sex"].str.upper() == "MALE"),
    "body_mass_g"
].dropna()

adelie_female = penguins_clean.loc[
    (penguins_clean["species"] == "Adelie") &
    (penguins_clean["sex"].str.upper() == "FEMALE"),
    "body_mass_g"
].dropna()

alpha = 0.05

# Complete aquí usando stats.ttest_ind(...).

## 10. Normalidad: prueba formal + diagnóstico gráfico

La prueba de Shapiro-Wilk contrasta:

\[
H_0:\text{los datos son compatibles con una distribución normal},
\]

\[
H_1:\text{los datos no siguen una distribución normal}.
\]

### Actividad 7

Evalúe la normalidad de `body_mass_g` en:

1. toda la base;
2. machos Adelie;
3. hembras Adelie.

Para cada caso:

- aplique Shapiro-Wilk;
- use \(\alpha=0.05\);
- construya un gráfico Q-Q;
- compare la conclusión formal con la evidencia gráfica.

### Pregunta de análisis

> ¿Por qué la distribución conjunta de `body_mass_g` puede apartarse de la normalidad aunque dentro de grupos más homogéneos el comportamiento sea distinto?

In [ ]:
# TODO 13. Pruebas de Shapiro-Wilk.
# Sugerencia: construya un diccionario con las tres muestras y recórralo con un ciclo.

In [ ]:
# TODO 14. Gráficos Q-Q.
# Puede usar:
# stats.probplot(serie, dist="norm", plot=plt)

# PARTE VI. Síntesis

## 11. Conclusión analítica

Redacte una conclusión de **máximo 8 líneas** que responda:

1. ¿qué problemas de calidad tenía la base?
2. ¿encontró evidencia de valores atípicos que debieran eliminarse?
3. ¿qué aprendió al separar los datos por especie?
4. ¿qué informó el IC sobre la masa corporal media de Gentoo?
5. ¿qué concluyó la prueba de Welch para Adelie?
6. ¿qué mostró el análisis de normalidad?

La conclusión debe incluir **evidencia numérica** y evitar afirmaciones causales.